<a href="https://colab.research.google.com/github/ammar-aa/Fly_rank_internship_repo/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [86]:
"""
Method: Pairwise ranking via Logistic Regression on feature differences

There is no ground-truth label in this problem. The Week 4 baseline (score = -trend_pct * position_share) is itself a hand-written scoring rule built from two signals — trend_pct and gsc_sum_position — not a target to predict against. So this isn't classification or regression toward a known truth; it's a comparison between two different ways of scoring and ranking the same rows.

What actually matters for this problem is the order pages fall in, not the exact score value — the baseline's real job is to rank pages by refresh urgency. Pairwise ranking directly optimizes for "does page A outrank page B," which matches that goal more precisely than trying to hit an arbitrary numeric score.

To keep the comparison fair, the model uses the same two signals the baseline formula uses (trend_pct, gsc_sum_position) — no additional features, no leakage.

Logistic Regression is used because it's the simplest model that can learn a combination of the two signals. The baseline combines them multiplicatively with fixed, hand-picked weighting (-trend_pct * position_share). Training a Logistic Regression on pairwise feature differences tests whether a different, learned combination of the same two signals produces a meaningfully different — and possibly more sensible — ranking than the fixed formula.

Other menu methods don't fit as well here: clustering isn't appropriate since the goal isn't to discover groups, it's to compare two ranking systems; and Decision Tree / Random Forest / Gradient Boosting are heavier than needed for two features and would obscure the direct, interpretable weight comparison against the formula's fixed coefficients.
"""

'\nSection 1: Method choice and why\n\nMethod: Pairwise ranking via Logistic Regression on feature differences\n\nThere is no ground-truth label in this problem. The Week 4 baseline (score = -trend_pct * position_share) is itself a hand-written scoring rule built from two signals — trend_pct and gsc_sum_position — not a target to predict against. So this isn\'t classification or regression toward a known truth; it\'s a comparison between two different ways of scoring and ranking the same rows.\n\nWhat actually matters for this problem is the order pages fall in, not the exact score value — the baseline\'s real job is to rank pages by refresh urgency. Pairwise ranking directly optimizes for "does page A outrank page B," which matches that goal more precisely than trying to hit an arbitrary numeric score.\n\nTo keep the comparison fair, the model uses the same two signals the baseline formula uses (trend_pct, gsc_sum_position) — no additional features, no leakage.\n\nLogistic Regression 

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [87]:
"""
Grouped by client, not time-aware.

Each content_hash_id appears exactly once in the dataset (verified: value_counts().max() == 1), so there is no repeated time series at the row level to be time-aware about. The Week 4 aggregation already collapsed the daily-grain source data into one summary row per page, with trend_pct and gsc_sum_position computed across the full time window per page. A time-aware split would be guarding against a leak that structurally cannot occur here, so it isn't used.

The real leak risk is client-level: multiple pages likely share the same client_hash_id, and if pages are split randomly, pages from the same client could land on both sides of train/test. Client-level effects (e.g. one client's whole site trending down for reasons unrelated to trend_pct or gsc_sum_position individually) could let the model partly learn "this client's pages behave a certain way" instead of the actual signal relationship being tested. Splitting by client_hash_id, so every page belonging to a given client stays entirely on one side, closes that leak.

Because this is a pairwise ranking setup, the split happens at the row level first, before pairs are generated: clients are divided into a train pool and a test pool, and only afterward are pairs sampled — separately — within each pool. This guarantees no single row, and no client, appears on both sides of any pair.
"""

'\nSection 2: Split design\n\nGrouped by client, not time-aware.\n\nEach content_hash_id appears exactly once in the dataset (verified: value_counts().max() == 1), so there is no repeated time series at the row level to be time-aware about. The Week 4 aggregation already collapsed the daily-grain source data into one summary row per page, with trend_pct and gsc_sum_position computed across the full time window per page. A time-aware split would be guarding against a leak that structurally cannot occur here, so it isn\'t used.\n\nThe real leak risk is client-level: multiple pages likely share the same client_hash_id, and if pages are split randomly, pages from the same client could land on both sides of train/test. Client-level effects (e.g. one client\'s whole site trending down for reasons unrelated to trend_pct or gsc_sum_position individually) could let the model partly learn "this client\'s pages behave a certain way" instead of the actual signal relationship being tested. Splittin

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [88]:
from google.colab import userdata
auth=userdata.get("HF_TOKEN")
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from scipy.stats import spearmanr
con=duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{auth}'
);
""")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

In [89]:
df = con.sql(f"""
SELECT *
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [90]:
dfF = con.sql(f"""
SELECT SUM(gsc_impressions) AS gsc_impressions, client_hash_id, content_hash_id
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet'
GROUP BY client_hash_id, content_hash_id
""").df()

dfM = con.sql(f"""
SELECT SUM(gsc_impressions) AS gsc_impressions, client_hash_id, content_hash_id
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
GROUP BY client_hash_id, content_hash_id
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [91]:
df_trend = dfM.merge(dfF, on=['client_hash_id', 'content_hash_id'], suffixes=('_feb', '_mar'), how='outer')

In [92]:
df_trend = df_trend[df_trend['gsc_impressions_feb'] >= 30]
df_trend = df_trend[df_trend['gsc_impressions_mar'] > 0]

df_trend['trend_pct'] = (
    (df_trend['gsc_impressions_mar'] - df_trend['gsc_impressions_feb'])
    / df_trend['gsc_impressions_feb']
) * 100

clip_value = df_trend['trend_pct'].quantile(0.99)
df_trend['trend_pct'] = df_trend['trend_pct'].clip(lower=-clip_value, upper=clip_value)

In [93]:
df = df.groupby(['client_hash_id', 'content_hash_id'], as_index=False).agg(
    gsc_sum_position=('gsc_sum_position', 'sum'),
    gsc_avg_position=('gsc_avg_position', 'mean'),
)

In [94]:
df = df.merge(df_trend[['client_hash_id', 'content_hash_id', 'trend_pct']], on=['client_hash_id', 'content_hash_id'], how='left')

In [95]:
negative_mean = df.loc[df['trend_pct'] < 0, 'trend_pct'].mean()
positive_mean = df.loc[df['trend_pct'] > 0, 'trend_pct'].mean()
conditions = [
    df['trend_pct'] < negative_mean,
    (df['trend_pct'] < 0) & (df['trend_pct'] >= negative_mean),
    (df['trend_pct'] >= 0) & (df['trend_pct'] < positive_mean),   # now includes 0
    df['trend_pct'] >= positive_mean
]
ranks = ['Sharp decline', 'Mild decline', 'Mild growth', 'Strong growth']
df['trend_dir'] = np.select(conditions, ranks, default=None)

In [96]:
position_share = df['gsc_sum_position'] / df['gsc_sum_position'].sum()

cap_value = position_share.quantile(0.99)
position_share_capped = position_share.clip(upper=cap_value)

score = -df['trend_pct'] * position_share_capped * 1000
df['score']=score
df['score'] = df['score'] * 1000

In [97]:
df = df[['content_hash_id', 'client_hash_id', 'trend_pct', 'gsc_sum_position', 'score']].copy()

In [98]:
conditions = [
    (df['trend_pct'] < negative_mean) & (position_share > position_share.median()),
    (df['trend_pct'] < negative_mean) & (position_share <= position_share.median()),
    (df['trend_pct'] >= negative_mean) & (df['trend_pct'] < 0) & (position_share > position_share.median()),
    (df['trend_pct'] >= negative_mean) & (df['trend_pct'] < 0),
    (df['trend_pct'] >= 0) & (position_share > position_share.median()),
]
codes = [
    'strong declining trend with low page position',
    'strong declining trend',
    'mild declining trend with low page position',
    'mild declining trend',
    'low page position',
]
df['reason_code'] = np.select(conditions, codes, default='STABLE')


In [99]:
df['action'] = np.select(
    [
        df['reason_code'] == 'strong declining trend with low page position',
        df['reason_code'].isin(['strong declining trend', 'mild declining trend with low page position']),
    ],
    ['REFRESH', 'MONITOR'],
    default='SKIP'
)

In [100]:
df = df.dropna(subset=['trend_pct', 'score']).reset_index(drop=True)
print("Rows after dropping NaN trend_pct/score:", len(df))

Rows after dropping NaN trend_pct/score: 105120


In [101]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_hash_id']))
df_train = df.iloc[train_idx].reset_index(drop=True)
df_test = df.iloc[test_idx].reset_index(drop=True)
print("Train rows:", len(df_train), "| Test rows:", len(df_test))
print("Client overlap (should be 0):", len(set(df_train['client_hash_id']) & set(df_test['client_hash_id'])))

Train rows: 99603 | Test rows: 5517
Client overlap (should be 0): 0


In [102]:
def make_balanced_pairs(data, n_pairs, seed, scaler=None, fit_scaler=False, cap_value=None):
    rng = np.random.default_rng(seed)
    idx_a = rng.integers(0, len(data), n_pairs)
    idx_b = rng.integers(0, len(data), n_pairs)
    mask = idx_a != idx_b
    idx_a, idx_b = idx_a[mask], idx_b[mask]

    trend = data['trend_pct'].values
    pos_capped = np.clip(data['gsc_sum_position'].values, a_min=None, a_max=cap_value)
    interaction = trend * pos_capped
    feats = np.column_stack([trend, pos_capped, interaction])

    if fit_scaler:
        scaler = StandardScaler()
        feats = scaler.fit_transform(feats)
    else:
        feats = scaler.transform(feats)

    scores = data['score'].values
    label = (scores[idx_a] > scores[idx_b]).astype(int)
    swap = label == 0
    idx_a_final = np.where(swap, idx_b, idx_a)
    idx_b_final = np.where(swap, idx_a, idx_b)

    X_diff = feats[idx_a_final] - feats[idx_b_final]
    y = np.ones(len(idx_a_final), dtype=int)
    flip = rng.random(len(y)) < 0.5
    X_diff[flip] = -X_diff[flip]
    y[flip] = 0

    return X_diff, y, scaler

cap_value = np.quantile(df_train['gsc_sum_position'].values, 0.99)

X_train, y_train, scaler = make_balanced_pairs(df_train, n_pairs=150_000, seed=42, fit_scaler=True, cap_value=cap_value)
X_test, y_test, _ = make_balanced_pairs(df_test, n_pairs=30_000, seed=99, scaler=scaler, fit_scaler=False, cap_value=cap_value)
print("Train label balance:", y_train.mean(), "| Test label balance:", y_test.mean())

Train label balance: 0.5000866678222377 | Test label balance: 0.49699959994665954


In [103]:
model = LogisticRegression()
model.fit(X_train, y_train)
train_acc = model.score(X_train, y_train)
test_acc = model.score(X_test, y_test)
print("Train pair accuracy:", train_acc)
print("Test pair accuracy:", test_acc)
print("Learned weights [trend_pct, gsc_sum_position, interaction]:", model.coef_[0])

Train pair accuracy: 0.979553060707476
Test pair accuracy: 0.964095212695026
Learned weights [trend_pct, gsc_sum_position, interaction]: [ -0.73185832  -0.36648833 -51.73579221]


In [104]:
pos_capped_full = np.clip(df['gsc_sum_position'].values, a_min=None, a_max=cap_value)
interaction_full = df['trend_pct'].values * pos_capped_full
feats_all = np.column_stack([df['trend_pct'].values, pos_capped_full, interaction_full])
feats_scaled = scaler.transform(feats_all)
df['model_score'] = feats_scaled @ model.coef_[0]


In [108]:
corr, pval = spearmanr(df['score'], df['model_score'])
print("Spearman correlation (baseline vs model):", corr)

comparison_rows = []
for k in [10, 100, 1000, 3000, 5000]:
    top_baseline = set(df.nlargest(k, 'score')['content_hash_id'])
    top_model = set(df.nlargest(k, 'model_score')['content_hash_id'])
    overlap = len(top_baseline & top_model) / k
    comparison_rows.append({'K': k, 'Top-K overlap': overlap})
    print(f"Top-{k} overlap: {overlap:.2%}")

comparison_table = pd.DataFrame([
    {'Metric': 'Test pairwise accuracy', 'Value': test_acc},
    {'Metric': 'Spearman correlation', 'Value': corr},
] + [{'Metric': f"Top-{r['K']} overlap", 'Value': r['Top-K overlap']} for r in comparison_rows])
comparison_table

Spearman correlation (baseline vs model): 0.9960958878776748
Top-10 overlap: 20.00%
Top-100 overlap: 46.00%
Top-1000 overlap: 57.30%
Top-3000 overlap: 85.73%
Top-5000 overlap: 95.54%


,Metric,Value
0,Test pairwise accuracy,0.964095
1,Spearman correlation,0.996096
2,Top-10 overlap,0.200000
3,Top-100 overlap,0.460000
4,Top-1000 overlap,0.573000
5,Top-3000 overlap,0.857333
6,Top-5000 overlap,0.955400


In [111]:
display(df.head(20).sort_values('score', ascending=False),df.head(20).sort_values('model_score', ascending=False))

,content_hash_id,client_hash_id,trend_pct,gsc_sum_position,score,reason_code,action,model_score
15,content_be06033d30b49299,client_0797ff3a1fc9a6a5,-26.051625,111186,892.237816,mild declining trend with low page position,MONITOR,25.600840
18,content_fa84e03e3d5e2fa2,client_0797ff3a1fc9a6a5,-33.281654,19567,200.597188,mild declining trend with low page position,MONITOR,-4.468553
3,content_1207efddce873942,client_0797ff3a1fc9a6a5,-62.255965,6679,128.082004,strong declining trend with low page position,REFRESH,-7.253697
6,content_27f8100281413b37,client_0797ff3a1fc9a6a5,-87.068966,4379,117.444864,strong declining trend with low page position,REFRESH,-7.405967
0,content_04c67f3541177192,client_0797ff3a1fc9a6a5,-25.679758,4759,37.644569,mild declining trend with low page position,MONITOR,-11.707309
4,content_167472cd0802a8f3,client_0797ff3a1fc9a6a5,-29.310345,2775,25.054147,mild declining trend with low page position,MONITOR,-12.209641
19,content_00014efc121d911d,client_08a6a72ff48e62c0,-90.517241,672,18.736837,strong declining trend with low page position,REFRESH,-11.721996
17,content_c89645311e3a5d17,client_0797ff3a1fc9a6a5,-84.848485,619,16.178209,strong declining trend with low page position,REFRESH,-11.905221
13,content_a0a6b37ae2f9a09c,client_0797ff3a1fc9a6a5,-8.870968,5388,14.722931,mild declining trend with low page position,MONITOR,-12.935659
7,content_2f719399052f18fc,client_0797ff3a1fc9a6a5,-47.222222,932,13.556826,mild declining trend with low page position,MONITOR,-12.487951


,content_hash_id,client_hash_id,trend_pct,gsc_sum_position,score,reason_code,action,model_score
15,content_be06033d30b49299,client_0797ff3a1fc9a6a5,-26.051625,111186,892.237816,mild declining trend with low page position,MONITOR,25.600840
18,content_fa84e03e3d5e2fa2,client_0797ff3a1fc9a6a5,-33.281654,19567,200.597188,mild declining trend with low page position,MONITOR,-4.468553
3,content_1207efddce873942,client_0797ff3a1fc9a6a5,-62.255965,6679,128.082004,strong declining trend with low page position,REFRESH,-7.253697
6,content_27f8100281413b37,client_0797ff3a1fc9a6a5,-87.068966,4379,117.444864,strong declining trend with low page position,REFRESH,-7.405967
0,content_04c67f3541177192,client_0797ff3a1fc9a6a5,-25.679758,4759,37.644569,mild declining trend with low page position,MONITOR,-11.707309
19,content_00014efc121d911d,client_08a6a72ff48e62c0,-90.517241,672,18.736837,strong declining trend with low page position,REFRESH,-11.721996
17,content_c89645311e3a5d17,client_0797ff3a1fc9a6a5,-84.848485,619,16.178209,strong declining trend with low page position,REFRESH,-11.905221
4,content_167472cd0802a8f3,client_0797ff3a1fc9a6a5,-29.310345,2775,25.054147,mild declining trend with low page position,MONITOR,-12.209641
7,content_2f719399052f18fc,client_0797ff3a1fc9a6a5,-47.222222,932,13.556826,mild declining trend with low page position,MONITOR,-12.487951
9,content_3cf5722aa6fd767f,client_0797ff3a1fc9a6a5,-56.250000,284,4.920810,strong declining trend with low page position,REFRESH,-12.755936


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
"""
The model largely converges on the baseline formula rather than independently discovering it.

Once the interaction term (trend_pct × capped gsc_sum_position) was added, its correlation with the baseline's own score came out to -0.92 — the two are almost mathematically the same quantity. Any model given that feature will lean on it heavily, because it's close to being a rescaled copy of the label. This explains the two clearest error patterns below.

Tree-based models (Decision Tree, Random Forest, Gradient Boosting) reached 100% pairwise test accuracy once given the interaction feature. This isn't genuine model quality — it's the models exploiting a feature that nearly duplicates the label. These results were excluded from the final comparison rather than reported, since a perfect score here reflects the feature's closeness to the target, not learned insight.

Linear models (Logistic Regression, Linear SVM) were more resistant to this, since their linear constraint prevents them from carving an exact boundary around one dominant feature the way trees can. Both still leaned heavily on the interaction term (its weight dwarfed trend_pct and gsc_sum_position in both models), but landed at more plausible accuracies (0.964 and 0.924 respectively) rather than a suspicious 1.000 — worth reading as the linear constraint acting as a mild, accidental safeguard against fully exploiting the near-duplicate feature.

Where the model and baseline actually disagree: the very top of the ranking, not the bulk of it. Top-K overlap climbs steeply with K across every model tested:

K	Logistic Regression	Linear SVM
10	20%	20%
100	46%	46%
1,000	57.3%	57.6%
3,000	85.7%	86.1%
5,000	95.5%	96.0%

This is the honest, specific answer to "where is the model wrong": both methods agree strongly on the broad set of pages worth attention (95%+ agreement by the top 5,000), but disagree sharply on the exact top 10–100 — the handful of pages that would actually get prioritized first in practice. That gap traces to a concrete, diagnosed cause, not vague imprecision: the baseline applies a 99th-percentile cap to position_share specifically to prevent a few extreme-volume pages from dominating the top of the ranking; the model, even after we capped gsc_sum_position consistently in feature construction, is still a linear/near-linear approximation of a formula that treats the cap and the multiplication together in a fixed way the model can only partially reproduce.

Two different linear model families (Logistic Regression, Linear SVM) converged on nearly identical top-K rankings despite different loss functions and different pairwise accuracy/Spearman numbers. This is a meaningful robustness check: the ranking result isn't an artifact of one specific optimizer — different linear methods, given the same three features, independently arrive at a similar ranking, which strengthens confidence that this is a real relationship in the data rather than a fitting quirk.

What the model leans on: across every legitimate result, the interaction term dominates by a wide margin over the two raw signals individually. This means the model's practical takeaway echoes the baseline's own design choice — the multiplicative interaction between trend and position carries far more information than either signal alone — but it does not validate that the baseline's specific formula (or its exact cap threshold) is correct, only that a similar structural combination fits well.
"""

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.